# Types

> Core internal types.

In [ ]:
#| default_exp types

## Imports

In [ ]:
#| export
import httpx
from dataclasses import dataclass, field
from fastcore.net import urljson
from fastcore.utils import *

In [ ]:
#| hide
from fastcore.test import *
import litellm

10:40:16 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'


10:40:16 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


The main philosophy of `fastllm` is to provide unified API to users where they can simply swap models/providers without code changes. The following canonical types are defined as internal represenatation:

## Part

Canonical atomic content unit for multimodal inputs/outputs.

**Why it exists**
- Providers represent content blocks differently (`input_text`, `text`, `inlineData`, `source`, etc).
- `Part` gives one stable internal shape so serialization/parsing logic can stay at provider boundaries.

**Design Notes**
- `type` encodes semantic modality (`text`, `input_image`, `input_file`, etc).
- `text` is for simple text payloads.
- `data` carries structured provider-agnostic payload details when text is insufficient.

**Connections**
- Wrapped by `Msg.content`.
- Produced/consumed by `highlevel` coercion, provider serializers in `clients`, and normalizers in `normalize`.
- Key enabler for model-only swapping without rewriting message construction code.

All four providers represent content parts differently in their wire format:

- **[OpenAI Responses](https://developers.openai.com/api/reference/resources/responses/methods/create)** — Content is an array of typed parts: `{"type": "input_text", "text": "..."}`, `{"type": "input_image", "image_url": "..."}`, `{"type": "input_audio", "input_audio": {"data": "...", "format": "..."}}`, `{"type": "input_file", "file_data": "data:application/pdf;base64,...", "filename": "..."}`
- **[OpenAI-compat (Chat)](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create)** — Content is an array of typed parts: `{"type": "text", "text": "..."}`, `{"type": "image_url", "image_url": {"url": "..."}}`, `{"type": "input_audio", "input_audio": {"data": "...", "format": "..."}}`, `{"type": "file", "file": {"file_data": "data:application/pdf;base64,...", "filename": "..."}}`
- **[Anthropic](https://docs.anthropic.com/en/api/messages)** — Content blocks with `source` nesting: `{"type": "text", "text": "..."}`, `{"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": "..."}}`, `{"type": "document", "source": {"type": "base64", "media_type": "application/pdf", "data": "..."}}`
- **[Gemini](https://ai.google.dev/api/generate-content)** — A `parts` array of `{text}` or `{inlineData: {mimeType, data}}` or `{fileData: {mimeType, fileUri}}`: `{"text": "..."}`, `{"inlineData": {"mimeType": "image/jpeg", "data": "..."}}`, `{"fileData": {"mimeType": "application/pdf", "fileUri": "..."}}`

> **Spec sources:** OpenAI Responses `Input{Text,Image,File}Content` (`specs/openai.with-code-samples.yml:68093–68444`), OpenAI Chat `ChatCompletionRequestMessageContentPart*` (`specs/openai.with-code-samples.yml:35185–35321`), Anthropic `Request{Text,Image,Document}Block` (`specs/anthropic.yml:12159–12458`), Gemini `Part`/`Blob`/`FileData` (`specs/gemini.json:189–387`).

`Part` canonicalizes these into a uniform `(type, text, data)` triple. The canonical types, their aliases, and expected input formats:

| Canonical `Part.type` | Aliases accepted | Expected input format | `Part.text` | `Part.data` |
|---|---|---|---|---|
| `"text"` | — | Plain string | `"the text"` | `None` |
| `"input_image"` | `image`, `image_url` | URL, data URL (`data:image/...;base64,...`), or provider-native `source`/`inlineData` in `data` | URL shorthand | `{"url": "..."}` or `{"image_url": "..."}` or provider-native payload |
| `"input_audio"` | `audio` | base64 data + format (OpenAI), URL/`fileUri` (Gemini). Not supported on Anthropic. | URL shorthand | `{"input_audio": {"data": "...", "format": "wav"}}` or `{"inlineData": {...}}` |
| `"input_video"` | `video`, `video_url` | URL or `fileUri`. Currently Gemini only; OpenAI maps to `input_file`. Not supported on Anthropic. | URL shorthand | `{"video_url": "..."}` or `{"fileData": {...}}` |
| `"input_file"` | `file`, `pdf`, `document` | data URL, URL, `file_id`, or provider-native `source` in `data` | URL shorthand | `{"file_data": "data:...;base64,...", "filename": "..."}` or `{"source": {...}}` |

**Provider support matrix:**

| Canonical type | OpenAI Responses | OpenAI-compat (Chat) | Anthropic | Gemini |
|---|---|---|---|---|
| `text` | ✅ `input_text` | ✅ `text` | ✅ `text` | ✅ `text` |
| `input_image` | ✅ `input_image` | ✅ `image_url` | ✅ `image` + `source` | ✅ `inlineData` / `fileData` |
| `input_audio` | ✅ `input_audio` | ✅ `input_audio` | ❌ (raises `UnsupportedCapabilityError`) | ✅ `inlineData` / `fileData` |
| `input_video` | ✅ (mapped to `input_file`) | ✅ (mapped to `input_file`) | ❌ (raises `UnsupportedCapabilityError`) | ✅ `fileData` |
| `input_file` | ✅ `input_file` | ✅ `file` | ✅ `document` + `source` | ✅ `inlineData` / `fileData` |

**Escape hatch:** For any type, setting `Part.data["<provider>"]` (e.g. `Part.data["anthropic"]`) to a dict bypasses canonicalization and passes the payload directly to that provider (see `_provider_part`).

**Canonicalization design:** `Part.data` uses OpenAI-style key conventions as the single canonical input format. Users write one shape (e.g. `{"image_url": "..."}`, `{"file_data": "data:...;base64,..."}`, `{"input_audio": {"data": "...", "format": "wav"}}`), and the provider serializers (`_openai_responses_part`, `_anthropic_part`, `_gemini_part`) dispatch these into provider-native wire formats. This means users can swap models/providers without changing their `Part` construction code. `Part.text` serves as a URL shorthand fallback — e.g. `Part(type="input_image", text="https://example.com/img.png")` works when you just have a URL and no other metadata.

The key insight: `type` carries the **semantic modality**, not the provider-specific type name. Provider-specific payload details live in `data`, keeping `Part` itself provider-agnostic. This means downstream code can branch on `Part.type` without knowing which provider produced it.

In [ ]:
#| export
@dataclass
class Part:
    "A normalized content part."
    type: str
    text: str = None
    data: dict = None

In [ ]:
#| export
PartType = str_enum('PartType', 'text', 'thinking', 'refusal', 'tool_use', 'server_tool_result', 'tool_result',
                    'input_image', 'input_audio', 'input_video', 'input_file')

In [ ]:
#| export
def _trunc_strs(o, n=200):
    "Truncate str or dict"
    if not o: return o
    if isinstance(o,str) and len(o)>n: return o[:100]+'...'
    if isinstance(o,dict): return {k: (v[:100]+'...' if isinstance(v,str) and len(v)>n else v) for k,v in o.items()}
    return o

@patch
def _repr_markdown_(self: Part):
    body = _trunc_strs(self.text) if self.text else ''
    data = _trunc_strs(self.data)
    return f"""**Part** (`{self.type}`)

{body}

<details markdown='1'>

- data: `{data}`

</details>"""

In [ ]:
Part(PartType.text, 'Hello world!'*1000, data={'long':"10"*1000})

<div class="prose" markdown="1">

**Part** (`text`)

Hello world!Hello world!Hello world!Hello world!Hello world!Hello world!Hello world!Hello world!Hell...

<details markdown='1'>

- data: `{'long': '1010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010...'}`

</details>

</div>

## Msg

Canonical conversation turn abstraction.

**Why it exists**
- Conversation structures vary across APIs (chat messages, response input items, content blocks).
- `Msg` lets the rest of fastllm reason in one conversation format.

**Design Notes**
- `role` captures turn semantics (`user`, `assistant`, `tool`, etc).
- `content` is a list of `Part` to support multimodal turns consistently.

**Connections**
- Primary input to `acompletion` and provider clients.
- Used by toolloop replay flows (`StreamSummary`/`Completion` -> `Msg` coercion in `highlevel`).

All three providers structure conversation messages differently:

- **[OpenAI Chat](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create)** — Messages are `{role, content}` objects. Roles: `system`, `developer`, `user`, `assistant`, `tool`. Content is a string or array of typed parts. Tool results use `role: "tool"` with `tool_call_id`.
- **[OpenAI Responses](https://developers.openai.com/api/reference/resources/responses/methods/create)** — Input items with `{role, content}`. Text parts become `input_text`/`output_text` depending on role. Tool results are `{type: "function_call_output", call_id, output}`.
- **[Anthropic](https://docs.anthropic.com/en/api/messages)** — Messages are `{role, content}` with roles `user` or `assistant` only. System prompt is a separate top-level parameter. Tool results are `tool_result` content blocks inside `role: "user"` messages.
- **[Gemini](https://ai.google.dev/api/generate-content)** — Messages are `{role, parts}`. Roles: `user` or `model`. System prompt uses `system_instruction`. Tool results are `functionResponse` parts inside `role: "user"` messages.

> **Spec sources:** OpenAI `ChatCompletionRequest*Message` (`specs/openai.with-code-samples.yml:35022–35444`), Anthropic `InputMessage` + `RequestToolResultBlock` (`specs/anthropic.yml:11140–11159,12595–12652`), Gemini `Content` (`specs/gemini.json:171–188`).

`Msg` normalizes these into a canonical `(role, content, data)` triple:

| Canonical `Msg.role` | OpenAI Chat | OpenAI Responses | Anthropic | Gemini |
|---|---|---|---|---|
| `"system"` | `role: "system"` | `role: "system"` | Separate `system` param | `system_instruction` |
| `"user"` | `role: "user"` | `role: "user"` | `role: "user"` | `role: "user"` |
| `"assistant"` | `role: "assistant"` | `role: "assistant"` | `role: "assistant"` | `role: "model"` |
| `"tool"` | `role: "tool"` + `tool_call_id` | `function_call_output` + `call_id` | `role: "user"` + `tool_result` block | `role: "user"` + `functionResponse` |

**`Msg.data` metadata:** Provider-agnostic metadata that doesn't fit `role`/`content`:
- `tool_calls` — list of tool call dicts (assistant messages)
- `tool_call_id` / `call_id` / `id` — tool call identifier (tool result messages)
- `name` — tool function name (tool result messages)
- `is_error` — whether the tool result is an error (Anthropic-specific, forwarded)

**Escape hatch:** Like `Part`, setting `Msg.data["<provider>"]` (e.g. `Msg.data["anthropic"]`) to a dict with `"role"` bypasses serialization entirely and passes the raw message to that provider.

In [ ]:
#| export
@dataclass
class Msg:
    "A normalized message."
    role: str
    content: List[Part]

    def _repr_markdown_(self):
        return f"""**Msg**

- role: `{self.role}`

<contents>

{'\n\n'.join(p._repr_markdown_() for p in self.content)}

</contents>"""

In [ ]:
Msg('user', content=[Part(PartType.text, 'Hello world!', data={'long':"10"*1000})]*3)

<div class="prose" markdown="1">

**Msg**

- role: `user`

<contents>

**Part** (`text`)

Hello world!

<details markdown='1'>

- data: `{'long': '1010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010...'}`

</details>

**Part** (`text`)

Hello world!

<details markdown='1'>

- data: `{'long': '1010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010...'}`

</details>

**Part** (`text`)

Hello world!

<details markdown='1'>

- data: `{'long': '1010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010...'}`

</details>

</contents>

</div>

## ToolCall

All four providers represent tool invocations differently in their wire format:

- **[OpenAI Responses](https://developers.openai.com/api/reference/resources/responses/methods/create)** — Tool calls are flat output items: `{type: "function_call", call_id, name, arguments}` where `arguments` is a JSON **string**. Streamed via `response.function_call_arguments.delta` events.
- **[OpenAI-compat (Chat)](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create)** — Tool calls are nested: `{id, type: "function", function: {name, arguments}}` where `arguments` is a JSON **string**. Streamed in chunks via `tool_calls[i].function.arguments` deltas.
- **[Anthropic](https://docs.anthropic.com/en/api/messages)** — Tool calls are `tool_use` content blocks: `{id, name, input, type: "tool_use"}` where `input` is a parsed JSON object. Streamed via `input_json_delta` events.
- **[Gemini](https://ai.google.dev/api/generate-content)** — Tool calls are `functionCall` parts: `{id, name, args}` where `args` is a parsed JSON object. Not chunked during streaming.

> **Spec sources:** OpenAI Responses `FunctionToolCall` (`specs/openai.with-code-samples.yml:44347–44389`), OpenAI Chat `ChatCompletionMessageToolCall` (`specs/openai.with-code-samples.yml:34869–34894`), Anthropic `ResponseToolUseBlock` (`specs/anthropic.yml:13563–13588`), Gemini `FunctionCall` (`specs/gemini.json:273–293`).

`ToolCall` canonicalizes these into a flat `(id, name, arguments)` triple where `arguments` is always a parsed `dict`:

| Field | OpenAI Responses | OpenAI-compat (Chat) | Anthropic | Gemini | `ToolCall` |
|---|---|---|---|---|---|
| ID | `call_id` | `id` | `id` | `id` | `id` |
| Name | `name` | `function.name` | `name` | `name` | `name` |
| Args | `arguments` (JSON string) | `function.arguments` (JSON string) | `input` (object) | `args` (object) | `arguments` (dict) |

In [ ]:
#| export
@dataclass
class ToolCall:
    "Normalized tool call."
    id: str
    name: str
    arguments: dict = field(default_factory=dict)
    server: bool = False
    extra: dict = field(default_factory=dict)

In [ ]:
#| export
@patch
def _repr_markdown_(self: ToolCall):
    extra = _trunc_strs(self.extra)
    return f"""🔧 **{self.name}**(`{self.arguments}`)

<details markdown='1'>

- id: `{self.id}`
- server: `{self.server}`
- extra: `{extra}`

</details>"""

def display_list(l): 
    from IPython.display import Markdown, display
    display(Markdown('\n\n'.join(o._repr_markdown_() for o in l)))

In [ ]:
ToolCall(id='oxwvx1fm', name='simple_add', arguments={'b': 547982745, 'a': 5478954793}, server=False, extra={'thoughtSignature': 'EscDCsQDAQw51scPHdv+D5BX7JWdLzz3Bv8tsKFRuAJe2UkTFZ+NZKzNsLtmQBiia+/r4HJEUptq1zQB0q9HToX0qzCUqyNAbDLY76KxMeW9jpsnUvh6ZjPM5sDD7fAafF7cjdApNMsihPqIZBAZjAlFPcp1c/50MObH5f1q7hO7fgDS4iSJ3Q3FfbAYWnJ4nlA2peVMu/6WFcKZh1wcZCIuN6iFCj6nhH+6RKkaFRaM0b6XCmpti6qldSeZx+qtHmo+lzr1tct4Gz/CITDI7gRJ3qfLYV2u45jOhKzdd1t6gQ39XLJ93j0xd0AwpzcdZLbHWqwWJCQ43nNzhJ7IQTAWOSyPgKDnlAMHq2PTEoXBYkMBApCZ1x+HncBzt77kQrTTe7sWGVmD5boVnYAIFPFGXOULP5tDZ+nog+Fg8NV10vaFKlHVf+VDzFnVWxT259LN12ykGtBilfpTXiKCV12RAZwhuL7vXXHrsBGg5HNVImcXqgMvwf/rtQlJeop+9bEcAiU48hMFMzumOrCmmHD3HgxpYLW7T3vtDmbNdKCDqVtIwO4Rp5HE6GudRWmq8iC2UnyQglUXoXVnxIZW7eYYDsGAYrYgZ1A='})

<div class="prose" markdown="1">

🔧 **simple_add**(`{'b': 547982745, 'a': 5478954793}`)

<details markdown='1'>

- id: `oxwvx1fm`
- server: `False`
- extra: `{'thoughtSignature': 'EscDCsQDAQw51scPHdv+D5BX7JWdLzz3Bv8tsKFRuAJe2UkTFZ+NZKzNsLtmQBiia+/r4HJEUptq1zQB0q9HToX0qzCUqyNAbDLY...'}`

</details>

</div>

## Usage

All four providers report token usage with different field names and structures:

- **[OpenAI Responses](https://developers.openai.com/api/reference/resources/responses/methods/create)** — `ResponseUsage`: `{input_tokens, output_tokens, total_tokens}` with `input_tokens_details.cached_tokens` and `output_tokens_details.reasoning_tokens`.
- **[OpenAI-compat (Chat)](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create)** — `CompletionUsage`: `{prompt_tokens, completion_tokens, total_tokens}` with `prompt_tokens_details.cached_tokens` and `completion_tokens_details.reasoning_tokens`.
- **[Anthropic](https://docs.anthropic.com/en/api/messages)** — `Usage`: `{input_tokens, output_tokens}` (no total) with `cache_read_input_tokens`, `cache_creation_input_tokens`, and `server_tool_use`.
- **[Gemini](https://ai.google.dev/api/generate-content)** — `UsageMetadata`: `{promptTokenCount, candidatesTokenCount, totalTokenCount}` with `cachedContentTokenCount`, `thoughtsTokenCount`, and `toolUsePromptTokenCount`.

> **Spec sources:** OpenAI Responses `ResponseUsage` (`specs/openai.with-code-samples.yml:62283–62317`), OpenAI Chat `CompletionUsage` (`specs/openai.with-code-samples.yml:36031–36080`), Anthropic `Usage` (`specs/anthropic.yml:14459–14508`), Gemini `UsageMetadata` (`specs/gemini.json:2200–2236`).

`Usage` normalizes these into a canonical `(prompt_tokens, completion_tokens, total_tokens, raw)` shape:

| Canonical field | OpenAI Responses | OpenAI-compat (Chat) | Anthropic | Gemini |
|---|---|---|---|---|
| `prompt_tokens` | `input_tokens` | `prompt_tokens` | `input_tokens` | `promptTokenCount` |
| `completion_tokens` | `output_tokens` | `completion_tokens` | `output_tokens` | `candidatesTokenCount` |
| `total_tokens` | `total_tokens` | `total_tokens` | (computed: in + out) | `totalTokenCount` |

**`Usage.raw` preserves the full provider payload** — advanced accounting fields (caching, reasoning, tool use tokens) vary too much across providers to normalize, so downstream code like `estimate_cost` extracts them from `raw` with best-effort multi-key lookups (e.g. `_cached_prompt_tokens` tries `cached_input_tokens`, `cache_read_input_tokens`, `prompt_tokens_details.cached_tokens`).

In [ ]:
#| export
@dataclass(frozen=True)
class Usage:
    "Normalized usage."
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_tokens: int = 0
    cached_tokens: int = 0
    cache_creation_tokens: int = 0
    reasoning_tokens: int = 0
    raw: dict = field(default_factory=dict)


## Completion

Canonical model response object

All four providers return non-stream responses in different shapes. `Completion` normalizes these into a canonical `(model, message, finish_reason, usage, tool_calls, raw)` object.

**Provider response → `Completion` field mapping:**

| Field | OpenAI Responses | OpenAI-compat (Chat) | Anthropic | Gemini |
|---|---|---|---|---|
| `model` | `raw.model` | `raw.model` | `raw.model` | passthrough (input model) |
| `message` | `output[].content[]` → `Msg(role="assistant", content=[Part...])` | `choices[0].message.content` → `Msg` | `content[]` → `Msg` | `candidates[0].content.parts[]` → text joined into single `Part` |
| `finish_reason` | `raw.status` (e.g. `"completed"`) | `choices[0].finish_reason` (e.g. `"stop"`) | `raw.stop_reason` (e.g. `"end_turn"`) | `candidates[0].finishReason` (e.g. `"STOP"`) |
| `usage` | `usage_from_openai(raw.usage)` | `usage_from_openai(raw.usage)` | `usage_from_anthropic(raw.usage)` | `usage_from_gemini(raw.usageMetadata)` |
| `tool_calls` | `output[]` where `type=="function_call"` → flat `ToolCall` | `message.tool_calls[].function` → `ToolCall` | `content[]` where `type=="tool_use"` → `ToolCall` | `parts[]` with `functionCall` → `ToolCall` |
| `raw` | full response dict | full response dict | full response dict | full response dict |

**Notable differences:**
- **Anthropic** puts tool calls in *both* `message.content` (as `Part(type="tool_use")`) *and* `tool_calls` — dual representation for downstream flexibility
- **Gemini** joins all text parts into a single string rather than preserving individual parts
- **Gemini** uses the input `model` string directly since the response only contains `modelVersion` (a version identifier, not a model name)

In [ ]:
#| export
@dataclass(frozen=True)
class Completion:
    "Normalized completion response."
    model: str
    message: Msg
    finish_reason: str = None
    usage: Usage = None
    tool_calls: List[ToolCall] = field(default_factory=list)
    api_name: str = None
    vendor_name: str = None
    raw: dict = field(default_factory=dict)

In [ ]:
#| export
@patch
def _repr_markdown_(self: Completion):
    message = self.message
    content = ''
    for p in message.content:
        if p.type == PartType.thinking:
            if p.text: content += f"<details><summary>Thinking</summary>\n\n{p.text}\n\n</details>\n\n"
        elif txt := p.text: content += txt
    if self.tool_calls:
        tool_calls = [f"\n\n🔧 {tc.name}({tc.arguments})\n" for tc in self.tool_calls]
        content += "\n".join(tool_calls)
    # for img in getattr(message, 'images', []): content += f"\n\n![generated image]({nested_idx(img, 'image_url', 'url')})" # TODO
    details = [f"model: `{self.model}`", f"finish_reason: `{self.finish_reason}`", f"usage: `{self.usage}`"]
    det_str = '\n- '.join(details)    
    return f"""{content}

<details markdown='1'>

- {det_str}

</details>"""

In [ ]:
parts = [
    Part(type=PartType.thinking, text="First, let me consider the question..."),
    Part(type=PartType.text, text="The answer involves two parts. "),
    Part(type=PartType.thinking, text="Now for the second part, I need to..."),
    Part(type=PartType.text, text="And here's the conclusion."),
]
Completion(model='model', message=Msg(role="assistant", content=parts))

<div class="prose" markdown="1">

<details><summary>Thinking</summary>

First, let me consider the question...

</details>

The answer involves two parts. <details><summary>Thinking</summary>

Now for the second part, I need to...

</details>

And here's the conclusion.

<details markdown='1'>

- model: `model`
- finish_reason: `None`
- usage: `None`

</details>

</div>

## Finish Reason

In [ ]:
#| export
FinishReason = str_enum('finish_reason', 'stop', 'tool_calls', 'length', 'content_filter')

## API Registry

In [ ]:
#| export
class APIRegistry:
    def __init__(self): self.apis = {}
    def register(self, name, finalize_usage=noop, **kwargs): self.apis[name] = SimpleNamespace(finalize_usage=finalize_usage, **kwargs)

api_registry = APIRegistry()


In [ ]:
def f(): print('test')
api_registry.register('test',**{'f':f})

In [ ]:
api = api_registry.apis['test']
api.f()

test


## Common Utils

In [ ]:
#| export
def mk_completion(resp, model, api_name, vendor_name):
    "Normalize an api response into Completion."
    api = api_registry.apis[api_name]
    tcs = api.norm_tool_calls(resp)
    parts = api.norm_parts(resp)
    usg = api.finalize_usage(api.norm_usage(resp), parts)
    return Completion(
        model=model,
        message=Msg(role="assistant", content=parts),
        finish_reason=api.norm_finish(resp, tcs),
        usage=usg,
        tool_calls=tcs,
        api_name=api_name,
        vendor_name=vendor_name,
        raw=resp)

In [ ]:
#| export
def mk_tool_res_msg(tool_calls:list[ToolCall], results:list[str|list]):
    'A util to prepare parallel tool call with str or media list results'
    parts = []
    for tc,res in zip(tool_calls, results):
        data = dict(id=tc.id, name=tc.name, arguments=tc.arguments, server=tc.server)
        parts.append(Part(type=PartType.tool_result, text=res, data=data))
    return Msg(role="tool", content=parts)

In [ ]:
#| export
def fn_schema(t):
    "Extract (name, description, parameters) from any tool format."
    if not isinstance(t, dict): return None
    fn = t.get('function')
    if isinstance(fn, dict): return fn.get('name',''), fn.get('description',''), fn.get('parameters',{})
    if 'name' in t and ('parameters' in t or 'input_schema' in t):
        return t.get('name',''), t.get('description',''), t.get('parameters', t.get('input_schema',{}))
    return None

In [ ]:
#| export
def sys_text(system):
    "Extract text from system (str or Part)."
    if system is None: return None
    return system if isinstance(system, str) else system.text

def part_txt(p): return p.text if isinstance(p,Part) else p

In [ ]:
#| export
@flexicache(time_policy(24*3600))
def _fetch_url_partial(url, nbytes=512): 
    "Fetch remote media bytes, optionally only first `nbytes`."
    try:
        with httpx.stream('GET', url, headers={'Range': f'bytes=0-{nbytes-1}'}, follow_redirects=True) as r:
            if r.status_code not in (200, 206): return
            return r.read()
    except (httpx.HTTPError, httpx.InvalidURL): return

In [ ]:
#| export
_ext_mime = {
    '.jpg':'image/jpeg', '.jpeg':'image/jpeg', '.png':'image/png', '.gif':'image/gif', '.webp':'image/webp',
    '.pdf':'application/pdf',
    '.mp3':'audio/mpeg', '.wav':'audio/wav', '.ogg':'audio/ogg', '.flac':'audio/flac', '.m4a':'audio/mp4',
    '.mp4':'video/mp4', '.mov':'video/quicktime', '.webm':'video/webm',
}

def data_url(url):
    "Parse data:mime;base64,data URL into (mime, b64_data), or None."
    if not isinstance(url, str) or not url.startswith('data:') or ',' not in url: return None
    header, body = url.split(',', 1)
    if ';base64' not in header or not body: return None
    return header[5:].split(';',1)[0].strip() or 'application/octet-stream', body

def url_mime(url, default='application/octet-stream'):
    "Guess mime from URL extension, and optional bytes fallback."
    if "youtube.com" in url or "youtu.be" in url: return "video/mp4"
    ext = '.' + url.rsplit('.', 1)[-1].split('?')[0].lower() if '.' in url.split('?')[0].split('/')[-1] else ''
    if (mime:=_ext_mime.get(ext)) is None: return detect_mime(_fetch_url_partial(url))
    return ifnone(mime, default)

In [ ]:
#| export
def payload_kwargs(msgs, model, stream=False, system=None, max_tokens=None, temperature=None, tools=None, tool_choice=None, reasoning_effort=None, web_search_options=None, stop_callables=None): pass

In [ ]:
#| export
def get_api_key(api_key, default):
    key = api_key or os.getenv(default)
    if not key: raise ValueError(f"Missing API key: set environment variable '{default}' or pass `api_key` parameter")
    return key

## Model Info and Cost

In [ ]:
#| export
model_prices_url = 'https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json'

@flexicache(time_policy(24*60*60))
def model_prices_meta(): return urljson(model_prices_url)

In [ ]:
#| export
def infer_api_name(model):
    "Infer api_name from model"
    if "claude" in model: return 'anthropic'
    if "gemini" in model: return 'gemini'
    if any(o in model for o in ('gpt','o3-','o4-')): return 'openai'

In [ ]:
#| export
@flexicache(time_policy(24*60*60))
def get_model_meta(model, vendor_name=None, tfm=noop):
    "Look up cost metadata for `model` from litellm price map, using `vendor_name` prefix if needed."
    vendor_name = ifnone(vendor_name, infer_api_name(model))
    mp, key = model_prices_meta(), ''
    if model in mp: key = model
    elif vendor_name=='gemini' and model.startswith('models/'): key = f"gemini/{model.removeprefix('models/')}"
    elif vendor_name:                                           key = f"{vendor_name}/{model}"
    return dict2obj(tfm(mp.get(key), model, vendor_name))

In [ ]:
#| export
haik45 = "claude-haiku-4-5"
sonn45 = "claude-sonnet-4-5"
sonn = sonn46 = "claude-sonnet-4-6"
opus46 = "claude-opus-4-6"
opus = "claude-opus-4-7"
gpt54 = "gpt-5.4"
gpt54m = "gpt-5.4-mini"
gpt55 = "gpt-5.5"
codex54 = "gpt-5.4"
codex54m = "gpt-5.4-mini"
codex55 = "gpt-5.5"
codex53spark = "gpt-5.3-codex-spark"

In [ ]:
#| export
model_info_registry = {}

def register_model_info(model, vendor_name=None, base=None, base_vendor_name=None, **overrides):
    "Register model metadata, optionally starting from `base`."
    info = dict(get_model_info(base, base_vendor_name or vendor_name)) if base else {}
    info.update(overrides)
    model_info_registry[vendor_name, model] = info

def get_model_info(mn, vendor_name=None):
    info = model_info_registry.get((vendor_name, mn)) or get_model_meta(mn, vendor_name)
    if 'search_context_cost_per_query' in info: info['supports_web_search'] = True
    return dict2obj(info)

In [ ]:
#| export
register_model_info('accounts/fireworks/models/qwen3p6-plus', vendor_name='fireworks_ai',
    supports_vision=True, supports_reasoning=True, supports_function_calling=True, supports_tool_choice=True,
    supports_system_messages=True, supports_response_schema=True, supports_parallel_function_calling=True,
    supports_prompt_caching=True, supports_native_streaming=True, supports_native_structured_output=True,
    max_tokens=1000000, max_input_tokens=1000000, max_output_tokens=65536,
    input_cost_per_token=0.5e-6, cache_read_input_token_cost=0.1e-6, output_cost_per_token=3.0e-6)

register_model_info('gemini-3.5-flash', vendor_name='gemini', base='gemini-3-flash-preview',
    input_cost_per_token=1.5e-6, output_cost_per_token=9e-6,
    output_cost_per_reasoning_token=9e-6, cache_read_input_token_cost=1.5e-7)

for model in ('gpt-5.4', 'gpt-5.4-mini'):
    register_model_info(model, vendor_name='openai', base=model, supports_web_search=True, mode=None)

for model in ('kimi-k2.5', 'kimi-k2.6'):
    register_model_info(model, vendor_name='moonshot', base=f'moonshot/{model}', base_vendor_name=None,
        supports_reasoning=True, supports_vision=True, supports_assistant_prefill=True)

register_model_info('gemini-3.1-flash-lite', vendor_name='gemini', base='gemini-3.1-flash-lite-preview')
register_model_info('models/gemini-3.1-flash-lite', vendor_name='gemini', base='gemini-3.1-flash-lite-preview')

for model in ('accounts/fireworks/models/kimi-k2p5', 'accounts/fireworks/models/kimi-k2p6'):
    register_model_info(model, vendor_name='fireworks_ai', base=model.replace('k2p6', 'k2p5'),
        supports_reasoning=True, supports_vision=True,
        input_cost_per_token=0.95e-6, cache_read_input_token_cost=0.16e-6, output_cost_per_token=4.0e-6)

In [ ]:
#| export
deepseek_v4_common = dict(
    supports_assistant_prefill=True, supports_function_calling=True, supports_prompt_caching=True,
    supports_reasoning=True, supports_tool_choice=True,
    max_input_tokens=1048576, max_output_tokens=393216, max_tokens=393216)

register_model_info('deepseek-v4-flash', vendor_name='deepseek', base='deepseek/deepseek-v3.2', **deepseek_v4_common,
    input_cost_per_token=1.4e-07, input_cost_per_token_cache_hit=2.8e-09,
    output_cost_per_token=2.8e-07, cache_read_input_token_cost=1.4e-07/10)
register_model_info('deepseek-v4-pro', vendor_name='deepseek', base='deepseek/deepseek-v3.2', **deepseek_v4_common,
    input_cost_per_token=4.35e-07, input_cost_per_token_cache_hit=3.625e-09,
    output_cost_per_token=8.7e-07, cache_read_input_token_cost=4.35e-07/10)

In [ ]:
#| export
codex_pricing = dict(
    input_cost_per_token = 0.10/1_000_000, output_cost_per_token = 0.50/1_000_000,
    cache_creation_input_token_cost = 0.10/1_000_000, cache_read_input_token_cost = 0.10/1_000_000)

for model in (codex54, codex54m, codex55):
    register_model_info(model, 'codex', base=model, base_vendor_name='chatgpt', supports_web_search=True, **codex_pricing)

register_model_info(codex53spark, 'codex', **codex_pricing,
    supports_vision=False, supports_image_input=False, supports_web_search=True, supports_reasoning=True,
    max_tokens=128000, max_input_tokens=128000, max_output_tokens=128000)


In [ ]:
get_model_info('deepseek-v4-flash', 'deepseek')

<div class="prose" markdown="1">

```python
{ 'cache_read_input_token_cost': 1.4000000000000001e-08,
  'input_cost_per_token': 1.4e-07,
  'input_cost_per_token_cache_hit': 2.8e-09,
  'litellm_provider': 'deepseek',
  'max_input_tokens': 1048576,
  'max_output_tokens': 393216,
  'max_tokens': 393216,
  'mode': 'chat',
  'output_cost_per_token': 2.8e-07,
  'supports_assistant_prefill': True,
  'supports_function_calling': True,
  'supports_prompt_caching': True,
  'supports_reasoning': True,
  'supports_tool_choice': True}
```

</div>

In [ ]:
def find_models(substr:str):
    "Find model names containing `substr`"
    return [k for k in model_prices_meta() if substr in k]

In [ ]:
'; '.join(find_models('sonnet-4-6'))

'anthropic.claude-sonnet-4-6; global.anthropic.claude-sonnet-4-6; us.anthropic.claude-sonnet-4-6; eu.anthropic.claude-sonnet-4-6; au.anthropic.claude-sonnet-4-6; jp.anthropic.claude-sonnet-4-6; azure_ai/claude-sonnet-4-6; claude-sonnet-4-6; vertex_ai/claude-sonnet-4-6; vertex_ai/claude-sonnet-4-6@default'

In [ ]:
#| export
def get_model_pricing(mn, vendor_name, million=True):
    return {k:round(v * (1e6 if million else 1), 6)
        for k,v in get_model_info(mn, vendor_name).items()
        if 'cost' in k and isinstance(v,float) and 'priority' not in k}

In [ ]:
get_model_pricing('claude-sonnet-4-6', 'anthropic')

{'cache_creation_input_token_cost': 3.75,
 'cache_read_input_token_cost': 0.3,
 'input_cost_per_token': 3.0,
 'output_cost_per_token': 15.0}

In [ ]:
get_model_pricing('gemini-3.5-flash', 'gemini')

{'cache_read_input_token_cost': 0.15,
 'input_cost_per_audio_token': 1.0,
 'input_cost_per_token': 1.5,
 'output_cost_per_reasoning_token': 9.0,
 'output_cost_per_token': 9.0}

In [ ]:
get_model_pricing('gemini-3.1-flash-lite', 'gemini')

{'cache_read_input_token_cost': 0.025,
 'cache_read_input_token_cost_per_audio_token': 0.05,
 'input_cost_per_audio_token': 0.5,
 'input_cost_per_token': 0.25,
 'output_cost_per_reasoning_token': 1.5,
 'output_cost_per_token': 1.5}

In [ ]:
#| export
def approx_pricing(nm, vendor_name, out=10, cache=80, inp=10, markup=0):
    "Approx cost per million tokens with given output/cache/input proportions"
    p = get_model_pricing(nm, vendor_name)
    ic = p.get('cache_creation_input_token_cost', p['input_cost_per_token'])
    res = (p['output_cost_per_token']*out + p['cache_read_input_token_cost']*cache + ic*inp) / (out+cache+inp)
    if nm=='claude-opus-4-7': res *= 1.5
    return res*(1+markup)

In [ ]:
for a,b in (('claude-opus-4-7', 'anthropic'),('claude-sonnet-4-6', 'anthropic'),(gpt55, 'openai'),(gpt54, 'openai'),(gpt54m, 'openai'),('gemini-3.5-flash', 'gemini'),('gemini-3.1-flash-lite', 'gemini'), ('accounts/fireworks/models/kimi-k2p6','fireworks_ai'), ('accounts/fireworks/models/kimi-k2p5','fireworks_ai')):
    print(f'${approx_pricing(a,b):.2f}', a,b)

$5.29 claude-opus-4-7 anthropic
$2.12 claude-sonnet-4-6 anthropic
$3.90 gpt-5.5 openai
$1.95 gpt-5.4 openai
$0.58 gpt-5.4-mini openai
$1.17 gemini-3.5-flash gemini
$0.20 gemini-3.1-flash-lite gemini
$0.62 accounts/fireworks/models/kimi-k2p6 fireworks_ai
$0.62 accounts/fireworks/models/kimi-k2p5 fireworks_ai


In [ ]:
#| export
@patch(as_prop=True)
def cost(self:Completion):
    meta = dict2obj(get_model_info(self.model, self.vendor_name))
    api = api_registry.apis[self.api_name]
    if not hasattr(api, 'cost'): raise NotImplementedError(f"API: {self.api_name} doesn't have a registered `cost` function in ns")
    return api.cost(self.usage, meta)

In [ ]:
get_model_info(codex53spark, 'codex')

<div class="prose" markdown="1">

```python
{ 'cache_creation_input_token_cost': 1.0000000000000001e-07,
  'cache_read_input_token_cost': 1.0000000000000001e-07,
  'input_cost_per_token': 1.0000000000000001e-07,
  'max_input_tokens': 128000,
  'max_output_tokens': 128000,
  'max_tokens': 128000,
  'output_cost_per_token': 5e-07,
  'supports_image_input': False,
  'supports_reasoning': True,
  'supports_vision': False,
  'supports_web_search': True}
```

</div>

In [ ]:
get_model_info('accounts/fireworks/models/qwen3p6-plus', 'fireworks_ai')

<div class="prose" markdown="1">

```python
{ 'cache_read_input_token_cost': 1e-07,
  'input_cost_per_token': 5e-07,
  'max_input_tokens': 1000000,
  'max_output_tokens': 65536,
  'max_tokens': 1000000,
  'output_cost_per_token': 3e-06,
  'supports_function_calling': True,
  'supports_native_streaming': True,
  'supports_native_structured_output': True,
  'supports_parallel_function_calling': True,
  'supports_prompt_caching': True,
  'supports_reasoning': True,
  'supports_response_schema': True,
  'supports_system_messages': True,
  'supports_tool_choice': True,
  'supports_vision': True}
```

</div>

In [ ]:
get_model_info('accounts/fireworks/models/kimi-k2p5', 'fireworks_ai')

<div class="prose" markdown="1">

```python
{ 'cache_read_input_token_cost': 1.6e-07,
  'input_cost_per_token': 9.5e-07,
  'litellm_provider': 'fireworks_ai',
  'max_input_tokens': 262144,
  'max_output_tokens': 262144,
  'max_tokens': 262144,
  'mode': 'chat',
  'output_cost_per_token': 4e-06,
  'source': 'https://fireworks.ai/pricing',
  'supports_function_calling': True,
  'supports_reasoning': True,
  'supports_response_schema': True,
  'supports_tool_choice': True,
  'supports_vision': True}
```

</div>

In [ ]:
get_model_info('gemini-3-pro-preview', 'gemini')

<div class="prose" markdown="1">

```python
{ 'cache_creation_input_token_cost_above_200k_tokens': 2.5e-07,
  'cache_read_input_token_cost': 2e-07,
  'cache_read_input_token_cost_above_200k_tokens': 4e-07,
  'cache_read_input_token_cost_above_200k_tokens_priority': 7.2e-07,
  'cache_read_input_token_cost_priority': 3.6e-07,
  'deprecation_date': '2026-03-26',
  'input_cost_per_token': 2e-06,
  'input_cost_per_token_above_200k_tokens': 4e-06,
  'input_cost_per_token_above_200k_tokens_priority': 7.2e-06,
  'input_cost_per_token_batches': 1e-06,
  'input_cost_per_token_priority': 3.6e-06,
  'litellm_provider': 'vertex_ai-language-models',
  'max_audio_length_hours': 8.4,
  'max_audio_per_prompt': 1,
  'max_images_per_prompt': 3000,
  'max_input_tokens': 1048576,
  'max_output_tokens': 65535,
  'max_pdf_size_mb': 30,
  'max_tokens': 65535,
  'max_video_length': 1,
  'max_videos_per_prompt': 10,
  'mode': 'chat',
  'output_cost_per_token': 1.2e-05,
  'output_cost_per_token_above_200k_tokens': 1.8e-05,
  'output_cost_per_token_above_200k_tokens_priority': 3.24e-05,
  'output_cost_per_token_batches': 6e-06,
  'output_cost_per_token_priority': 2.16e-05,
  'search_context_cost_per_query': { 'search_context_size_high': 0.014,
                                     'search_context_size_low': 0.014,
                                     'search_context_size_medium': 0.014},
  'source': 'https://cloud.google.com/vertex-ai/generative-ai/pricing',
  'supported_endpoints': ['/v1/chat/completions', '/v1/completions', '/v1/batch'],
  'supported_modalities': ['text', 'image', 'audio', 'video'],
  'supported_output_modalities': ['text'],
  'supports_audio_input': True,
  'supports_function_calling': True,
  'supports_native_streaming': True,
  'supports_pdf_input': True,
  'supports_prompt_caching': True,
  'supports_reasoning': True,
  'supports_response_schema': True,
  'supports_service_tier': True,
  'supports_system_messages': True,
  'supports_tool_choice': True,
  'supports_video_input': True,
  'supports_vision': True,
  'supports_web_search': True,
  'web_search_billing_unit': 'per_query'}
```

</div>

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()